# 07b — Semantic Vector RAG Pipeline

Replaces the original ID-based lookup with **true semantic retrieval** using vector embeddings.

### How it works
1. Embed all 1,261 clinical summaries into vectors using `sentence-transformers`
2. Store in a local **ChromaDB** vector database
3. For each of the 4,508 HF patients, convert their clinical profile → text → vector
4. Query ChromaDB for top-3 most semantically similar historical cases
5. Build enriched prompts combining: risk score + SHAP drivers + medication context + similar cases
6. Save to `rag_prompts_semantic.json`

### Coverage improvement
| Method | Coverage |
|---|---|
| Old ID matching | 26% (1,192 / 4,508) |
| **Semantic RAG** | **100% (4,508 / 4,508)** |

In [1]:
# Install required libraries
%pip install sentence-transformers chromadb -q

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


In [2]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))

import json
import pickle
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from utils import get_db_connection

DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'dataset')
ROOT_PATH = os.path.dirname(os.getcwd())

print('Libraries loaded successfully')

c:\Users\Kiruba\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Libraries loaded successfully


## Step 1 — Load Clinical Summaries
These are the 1,261 AI-generated clinical narratives from `RAG_data_summary.json` that will form our vector database.

In [3]:
# Load RAG summaries
with open(os.path.join(DATA_PATH, 'RAG_data_summary.json')) as f:
    rag_raw = json.load(f)

# Each record is a dict with one key = subject_id, value = clinical summary
rag_docs = []
for record in rag_raw:
    subject_id = list(record.keys())[0]
    summary    = list(record.values())[0]
    # Clean up the 'assistant\n\n' prefix if present
    summary = summary.replace('assistant\n\n', '').strip()
    rag_docs.append({
        'subject_id': str(subject_id),
        'summary':    summary
    })

print(f'Clinical summaries loaded: {len(rag_docs)}')
print(f'\nSample summary (first 300 chars):')
print(rag_docs[0]['summary'][:300])

Clinical summaries loaded: 1261

Sample summary (first 300 chars):
A 60-year-old male patient with a history of colon cancer, hypertension, and dyslipidemia presented with one week of dyspnea on exertion. Physical examination and initial lab results showed atrial flutter, elevated D-dimer levels, and a slightly elevated blood pressure. However, imaging tests did no


## Step 2 — Load Embedding Model
`all-MiniLM-L6-v2` is a lightweight but powerful model:
- Downloads once (~80MB), then runs locally
- Produces 384-dimensional vectors
- Fast: embeds 1,261 documents in ~30 seconds on CPU

In [4]:
print('Loading sentence-transformers model (downloads ~80MB on first run)...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded: all-MiniLM-L6-v2')
print(f'Embedding dimension: {embedder.get_sentence_embedding_dimension()}')

Loading sentence-transformers model (downloads ~80MB on first run)...
Model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


## Step 3 — Build ChromaDB Vector Database
Embed all 1,261 summaries and store in a persistent local ChromaDB collection.

In [5]:
CHROMA_PATH = os.path.join(ROOT_PATH, 'dataset', 'chroma_db')
os.makedirs(CHROMA_PATH, exist_ok=True)

# Persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete existing collection if re-running
try:
    chroma_client.delete_collection('hf_clinical_summaries')
    print('Deleted existing collection')
except:
    pass

collection = chroma_client.create_collection(
    name='hf_clinical_summaries',
    metadata={'hnsw:space': 'cosine'}  # cosine similarity for text
)

print(f'ChromaDB initialized at: {CHROMA_PATH}')
print('Embedding 1,261 clinical summaries (may take ~1-2 min on CPU)...')

# Embed in batches of 64 for memory efficiency
BATCH_SIZE = 64
texts      = [d['summary']    for d in rag_docs]
ids        = [d['subject_id'] for d in rag_docs]

for i in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[i:i+BATCH_SIZE]
    batch_ids   = ids[i:i+BATCH_SIZE]
    batch_embs  = embedder.encode(batch_texts, show_progress_bar=False).tolist()

    collection.add(
        documents=batch_texts,
        embeddings=batch_embs,
        ids=batch_ids,
        metadatas=[{'subject_id': sid} for sid in batch_ids]
    )

    if (i // BATCH_SIZE + 1) % 5 == 0:
        print(f'  Embedded {min(i+BATCH_SIZE, len(texts))}/{len(texts)} summaries...')

print(f'\nChromaDB collection built: {collection.count()} documents indexed')

ChromaDB initialized at: d:\Capstone project\Capstone_Healthcare_Decision_Intelligence_Agent\dataset\chroma_db
Embedding 1,261 clinical summaries (may take ~1-2 min on CPU)...
  Embedded 320/1261 summaries...
  Embedded 640/1261 summaries...
  Embedded 960/1261 summaries...
  Embedded 1261/1261 summaries...

ChromaDB collection built: 1261 documents indexed


## Step 4 — Load Patient Data + Model Artifacts
We need SHAP values and medication features for all 4,508 patients.

In [6]:
import shap

# Load model artifacts
artifacts_path = os.path.join(ROOT_PATH, 'model_artifacts.pkl')
with open(artifacts_path, 'rb') as f:
    artifacts = pickle.load(f)

scaler        = artifacts['scaler']
selector      = artifacts['selector']
lr_model      = artifacts['model']
feature_names = artifacts['selected_features']

print(f'Model loaded: {artifacts.get("model_type", "LR")}')
print(f'Features    : {len(feature_names)}')

# Load full cohort features from DuckDB
con = get_db_connection()
model_features = con.execute('SELECT * FROM model_features').fetchdf()
hf_labeled     = con.execute('SELECT hadm_id, subject_id, readmitted_30d FROM hf_labeled').fetchdf()
con.close()

print(f'\nCohort loaded: {len(model_features):,} admissions')

Model loaded: LR
Features    : 40
Connected to DuckDB at: d:\Capstone project\Capstone_Healthcare_Decision_Intelligence_Agent\dataset\hf_project.duckdb

Cohort loaded: 4,508 admissions


## Step 5 — Compute SHAP for All 4,508 Patients

In [7]:
import warnings, shap
warnings.filterwarnings('ignore')

# ── Get exact column names selector was fit on ────────────────────────────────
SELECTOR_COLS = selector.feature_names_in_.tolist()  # exact 146 columns from training

# ── Prepare raw features ──────────────────────────────────────────────────────
drop_cols = ['hadm_id', 'subject_id', 'readmitted_30d']
X_all_raw = model_features.drop(columns=[c for c in drop_cols if c in model_features.columns]).copy()

# Add lab change features
LABS = ['creatinine','urea_nitrogen','sodium','potassium','glucose','hemoglobin',
        'white_blood_cells','platelet_count','bicarbonate','calcium_total',
        'inrpt','ptt','troponin_t','creatine_kinase_mb_isoenzyme']
for lab in LABS:
    chg = f'{lab}_change'
    if chg not in X_all_raw.columns and f'{lab}_last' in X_all_raw.columns:
        X_all_raw[chg] = X_all_raw[f'{lab}_last'] - X_all_raw[f'{lab}_first']

# Impute nulls before encoding
num_cols = X_all_raw.select_dtypes(include=[np.number]).columns
for col in num_cols:
    X_all_raw[col] = X_all_raw[col].fillna(X_all_raw[col].median())

# One-hot encode categoricals — fill NA with UNKNOWN first (matches training)
cat_cols    = ['gender','admission_type','insurance','marital_status','race','discharge_location']
cat_present = [c for c in cat_cols if c in X_all_raw.columns]
for col in cat_present:
    X_all_raw[col] = X_all_raw[col].fillna('UNKNOWN')
X_all_raw = pd.get_dummies(X_all_raw, columns=cat_present, drop_first=True)

# ── Align to EXACT 146 columns selector was trained on ───────────────────────
# Add any missing columns as 0 (e.g. _UNKNOWN categories that didn't appear)
for col in SELECTOR_COLS:
    if col not in X_all_raw.columns:
        X_all_raw[col] = 0

# Keep only the 146 selector columns, in the exact same order
X_all_raw = X_all_raw[SELECTOR_COLS]

print(f'Feature matrix shape  : {X_all_raw.shape}')
print(f'Expected columns      : {len(SELECTOR_COLS)}')
print(f'Columns match         : {list(X_all_raw.columns) == SELECTOR_COLS}')

# ── Apply selector + scaler ───────────────────────────────────────────────────
X_all_selected = selector.transform(X_all_raw)
X_all_scaled   = scaler.transform(X_all_selected)

# ── Compute SHAP values ───────────────────────────────────────────────────────
print('\nComputing SHAP values for all patients...')
explainer   = shap.LinearExplainer(lr_model, X_all_scaled)
shap_values = explainer.shap_values(X_all_scaled)

# Predicted risk scores
risk_scores = lr_model.predict_proba(X_all_scaled)[:, 1]

print(f'SHAP computed for {len(risk_scores):,} patients')
print(f'Risk score range: {risk_scores.min()*100:.1f}% — {risk_scores.max()*100:.1f}%')


Feature matrix shape  : (4508, 137)
Expected columns      : 137
Columns match         : True

Computing SHAP values for all patients...
SHAP computed for 4,508 patients
Risk score range: 0.1% — 93.2%


## Step 6 — Build Patient Text Profiles
Convert each patient's clinical features into a natural language description for semantic search.

In [8]:
MED_KEYWORDS = ['loop','ace_arb','beta','aldo','digoxin','anticoag','unique_drugs','furosemide','gdmt']

def get_top_shap_drivers(shap_row, feature_names, n=3):
    """Return top n positive SHAP drivers (features increasing readmission risk)."""
    pos_idx  = np.where(shap_row > 0)[0]
    if len(pos_idx) == 0:
        pos_idx = np.argsort(-np.abs(shap_row))[:n]
    sorted_idx = pos_idx[np.argsort(-shap_row[pos_idx])][:n]
    return [(feature_names[i], float(shap_row[i])) for i in sorted_idx]

def build_patient_text(row, shap_row, risk_pct, feature_names):
    """Convert patient features into a natural language profile for embedding."""
    parts = []

    # Demographics
    age = row.get('age', 'unknown')
    los = row.get('length_of_stay', 'unknown')
    parts.append(f"Heart failure patient, age {age}, hospital stay {los} days.")

    # Risk level
    risk_label = 'HIGH' if risk_pct >= 50 else ('MEDIUM' if risk_pct >= 25 else 'LOW')
    parts.append(f"Predicted 30-day readmission risk: {risk_pct:.1f}% ({risk_label}).")

    # Key lab values
    lab_parts = []
    for lab, label in [('creatinine_last','creatinine'), ('bicarbonate_last','bicarbonate'),
                       ('hemoglobin_last','hemoglobin'), ('sodium_last','sodium')]:
        val = row.get(lab)
        if val and not np.isnan(float(val)):
            lab_parts.append(f"{label} {float(val):.1f}")
    if lab_parts:
        parts.append(f"Lab values at discharge: {', '.join(lab_parts)}.")

    # Medication context
    med_parts = []
    if row.get('on_loop_diuretic') == 1:
        dose = row.get('furosemide_max_dose', 0)
        med_parts.append(f"on loop diuretic (furosemide max dose {float(dose):.0f}mg)")
    else:
        med_parts.append("NOT on loop diuretic")
    if row.get('on_ace_arb') == 1:
        med_parts.append("on ACE inhibitor or ARB")
    else:
        med_parts.append("NOT on ACE inhibitor/ARB")
    if row.get('on_beta_blocker') == 1:
        med_parts.append("on beta-blocker")
    else:
        med_parts.append("NOT on beta-blocker")
    gdmt = row.get('gdmt_score', 0)
    med_parts.append(f"GDMT score {int(gdmt)}/4")
    parts.append(f"Medications: {'; '.join(med_parts)}.")

    # Top SHAP risk drivers
    top_drivers = get_top_shap_drivers(shap_row, feature_names, n=3)
    driver_strs = [f"{name.replace('_',' ')} (impact {val:+.3f})" for name, val in top_drivers]
    parts.append(f"Top risk drivers: {', '.join(driver_strs)}.")

    # Clinical flags
    flags = []
    if row.get('has_infection') == 1:
        flags.append("infection present")
    if row.get('num_prior_admissions', 0) > 1:
        flags.append(f"{int(row['num_prior_admissions'])} prior admissions")
    if row.get('num_unique_drugs', 0) > 20:
        flags.append(f"polypharmacy ({int(row['num_unique_drugs'])} drugs)")
    if flags:
        parts.append(f"Clinical flags: {', '.join(flags)}.")

    return ' '.join(parts)

# Build text profile for every patient
print('Building patient text profiles...')
patient_profiles = []
mf_dict = model_features.to_dict('records')

for i, row in enumerate(mf_dict):
    text = build_patient_text(row, shap_values[i], risk_scores[i]*100, feature_names)
    patient_profiles.append({
        'hadm_id':    row['hadm_id'],
        'subject_id': str(int(row['subject_id'])),
        'risk_pct':   round(risk_scores[i]*100, 1),
        'text':       text,
        'shap_row':   shap_values[i]
    })

print(f'Built {len(patient_profiles):,} patient text profiles')
print(f'\nSample profile:')
print(patient_profiles[0]['text'])

Building patient text profiles...
Built 4,508 patient text profiles

Sample profile:
Heart failure patient, age 40, hospital stay 2 days. Predicted 30-day readmission risk: 32.3% (MEDIUM). Lab values at discharge: creatinine 0.8, bicarbonate 23.0, hemoglobin 13.9, sodium 136.0. Medications: NOT on loop diuretic; NOT on ACE inhibitor/ARB; NOT on beta-blocker; GDMT score 0/4. Top risk drivers: hemoglobin min (impact +0.400), urea nitrogen max (impact +0.162), urea nitrogen first (impact +0.146).


## Step 7 — Semantic Retrieval for Each Patient
For each of the 4,508 patients, embed their profile and query ChromaDB for the top-3 most similar historical cases.

In [9]:
def retrieve_similar_cases(patient_text, collection, embedder, n_results=3):
    """Embed patient text and retrieve top-n similar clinical summaries."""
    query_emb = embedder.encode([patient_text]).tolist()
    results   = collection.query(
        query_embeddings=query_emb,
        n_results=n_results,
        include=['documents', 'distances', 'metadatas']
    )
    cases = []
    for doc, dist, meta in zip(
        results['documents'][0],
        results['distances'][0],
        results['metadatas'][0]
    ):
        similarity = round((1 - dist) * 100, 1)  # cosine distance → similarity %
        cases.append({
            'subject_id': meta['subject_id'],
            'similarity': similarity,
            'summary':    doc[:400]  # truncate for prompt brevity
        })
    return cases

# Test on one patient first
test_patient = patient_profiles[0]
similar = retrieve_similar_cases(test_patient['text'], collection, embedder)

print('=== SEMANTIC RETRIEVAL TEST ===')
print(f'Query patient hadm_id : {test_patient["hadm_id"]}')
print(f'Query patient risk    : {test_patient["risk_pct"]}%')
print(f'Query text            : {test_patient["text"][:200]}...')
print()
print('Top 3 Similar Cases Retrieved:')
for i, case in enumerate(similar, 1):
    print(f'\n  Case {i} (similarity: {case["similarity"]}%)')
    print(f'  Subject ID: {case["subject_id"]}')
    print(f'  Summary   : {case["summary"][:200]}...')

=== SEMANTIC RETRIEVAL TEST ===
Query patient hadm_id : 28334412
Query patient risk    : 32.3%
Query text            : Heart failure patient, age 40, hospital stay 2 days. Predicted 30-day readmission risk: 32.3% (MEDIUM). Lab values at discharge: creatinine 0.8, bicarbonate 23.0, hemoglobin 13.9, sodium 136.0. Medica...

Top 3 Similar Cases Retrieved:

  Case 1 (similarity: 61.6%)
  Subject ID: 24251291
  Summary   : The patient is a woman with a complex medical history, including diabetes, hypertension, and non-ischemic systolic heart failure with a severely decreased ejection fraction. She presented with symptom...

  Case 2 (similarity: 61.6%)
  Subject ID: 25335457
  Summary   : The patient, a 45-year-old male with a history of systolic congestive heart failure, coronary artery disease, and atrial fibrillation, presents with worsening dyspnea and lower extremity edema. Recent...

  Case 3 (similarity: 61.2%)
  Subject ID: 28129271
  Summary   : The patient is a 40-year-old male wi

## Step 8 — Build Enriched Prompts for All 4,508 Patients

In [10]:
def build_enriched_prompt(patient_profile, similar_cases, feature_names):
    """Build a rich LLM prompt combining patient data + similar historical cases."""
    shap_row = patient_profile['shap_row']
    risk_pct = patient_profile['risk_pct']

    # Top risk drivers
    top_drivers = get_top_shap_drivers(shap_row, feature_names, n=4)
    driver_lines = '\n'.join([
        f"  - {name.replace('_',' ').title()}: impact {val:+.3f}"
        for name, val in top_drivers
    ])

    # Similar cases section
    similar_section = ''
    for i, case in enumerate(similar_cases, 1):
        similar_section += (
            f"\n  Case {i} ({case['similarity']}% similar):\n"
            f"  {case['summary'][:300]}\n"
        )

    prompt = f"""--- CURRENT PATIENT PROFILE ---
{patient_profile['text']}

--- ML MODEL ASSESSMENT ---
Predicted 30-day readmission risk: {risk_pct:.1f}%
Risk level: {'HIGH' if risk_pct >= 50 else ('MEDIUM' if risk_pct >= 25 else 'LOW')}

Top contributing risk factors (SHAP analysis):
{driver_lines}

--- SIMILAR HISTORICAL CASES (Semantic Retrieval) ---
{similar_section}
--- TASK ---
Based on the patient profile, ML risk assessment, and similar historical cases above,
provide a structured clinical decision support response."""

    return prompt

# Build prompts for ALL patients
print('Building enriched prompts for all 4,508 patients...')
print('(Embedding + retrieval for each patient — may take 5-10 min)')

semantic_prompts = []
EMBED_BATCH = 128  # embed patient texts in batches for speed

# Batch embed all patient texts
all_texts = [p['text'] for p in patient_profiles]
print('Embedding all patient profiles...')
all_embeddings = embedder.encode(all_texts, batch_size=EMBED_BATCH,
                                  show_progress_bar=True)

print('Retrieving similar cases for each patient...')
for i, (profile, emb) in enumerate(zip(patient_profiles, all_embeddings)):
    # Query ChromaDB with pre-computed embedding
    results = collection.query(
        query_embeddings=[emb.tolist()],
        n_results=3,
        include=['documents', 'distances', 'metadatas']
    )

    similar_cases = []
    for doc, dist, meta in zip(
        results['documents'][0],
        results['distances'][0],
        results['metadatas'][0]
    ):
        similar_cases.append({
            'subject_id': meta['subject_id'],
            'similarity': round((1 - dist) * 100, 1),
            'summary':    doc[:400]
        })

    prompt = build_enriched_prompt(profile, similar_cases, feature_names)

    semantic_prompts.append({
        'hadm_id':       int(profile['hadm_id']),
        'risk_pct':      profile['risk_pct'],
        'prompt_context': prompt,
        'similar_cases': similar_cases
    })

    if (i + 1) % 500 == 0:
        print(f'  Processed {i+1:,} / {len(patient_profiles):,} patients')

print(f'\nTotal enriched prompts built: {len(semantic_prompts):,}')

Building enriched prompts for all 4,508 patients...
(Embedding + retrieval for each patient — may take 5-10 min)
Embedding all patient profiles...


Batches: 100%|██████████| 36/36 [01:28<00:00,  2.45s/it]


Retrieving similar cases for each patient...
  Processed 500 / 4,508 patients
  Processed 1,000 / 4,508 patients
  Processed 1,500 / 4,508 patients
  Processed 2,000 / 4,508 patients
  Processed 2,500 / 4,508 patients
  Processed 3,000 / 4,508 patients
  Processed 3,500 / 4,508 patients
  Processed 4,000 / 4,508 patients
  Processed 4,500 / 4,508 patients

Total enriched prompts built: 4,508


## Step 9 — Save Outputs

In [11]:
# Save semantic prompts (replaces old rag_prompts.json)
output_path = os.path.join(ROOT_PATH, 'rag_prompts_semantic.json')

# Convert numpy types for JSON serialization
def make_serializable(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

save_prompts = []
for p in semantic_prompts:
    save_prompts.append({
        'hadm_id':        make_serializable(p['hadm_id']),
        'risk_pct':       make_serializable(p['risk_pct']),
        'prompt_context': p['prompt_context'],
        'similar_cases':  p['similar_cases']
    })

with open(output_path, 'w') as f:
    json.dump(save_prompts, f, indent=2)

print(f'Saved rag_prompts_semantic.json')
print(f'  Total patients: {len(save_prompts):,}')
print(f'  Coverage      : 100% (was 26% with ID matching)')
print(f'  File size     : {os.path.getsize(output_path)/1024/1024:.1f} MB')
print()

# Show a sample prompt
sample = save_prompts[0]
print('=== SAMPLE ENRICHED PROMPT ===')
print(f'hadm_id  : {sample["hadm_id"]}')
print(f'risk_pct : {sample["risk_pct"]}%')
print(f'Similar cases similarity scores: {[c["similarity"] for c in sample["similar_cases"]]}')
print()
print(sample['prompt_context'][:1000])
print('...')

Saved rag_prompts_semantic.json
  Total patients: 4,508
  Coverage      : 100% (was 26% with ID matching)
  File size     : 15.6 MB

=== SAMPLE ENRICHED PROMPT ===
hadm_id  : 28334412
risk_pct : 32.3%
Similar cases similarity scores: [61.6, 61.6, 61.2]

--- CURRENT PATIENT PROFILE ---
Heart failure patient, age 40, hospital stay 2 days. Predicted 30-day readmission risk: 32.3% (MEDIUM). Lab values at discharge: creatinine 0.8, bicarbonate 23.0, hemoglobin 13.9, sodium 136.0. Medications: NOT on loop diuretic; NOT on ACE inhibitor/ARB; NOT on beta-blocker; GDMT score 0/4. Top risk drivers: hemoglobin min (impact +0.400), urea nitrogen max (impact +0.162), urea nitrogen first (impact +0.146).

--- ML MODEL ASSESSMENT ---
Predicted 30-day readmission risk: 32.3%
Risk level: MEDIUM

Top contributing risk factors (SHAP analysis):
  - Hemoglobin Min: impact +0.400
  - Urea Nitrogen Max: impact +0.162
  - Urea Nitrogen First: impact +0.146
  - Creatinine Last: impact +0.105

--- SIMILAR HISTO

In [12]:
# Final summary
print('='*60)
print('SEMANTIC RAG PIPELINE COMPLETE')
print('='*60)
print(f'  ChromaDB location      : dataset/chroma_db/')
print(f'  Documents indexed      : {collection.count():,}')
print(f'  Patients with prompts  : {len(semantic_prompts):,} (100% coverage)')
print(f'  Old coverage (ID match): 1,192 (26%)')
print(f'  Output file            : rag_prompts_semantic.json')
print()

# Similarity score distribution
all_similarities = [c['similarity'] for p in semantic_prompts for c in p['similar_cases']]
print(f'  Retrieval quality (similarity scores):')
print(f'    Mean similarity  : {np.mean(all_similarities):.1f}%')
print(f'    Median similarity: {np.median(all_similarities):.1f}%')
print(f'    Min similarity   : {np.min(all_similarities):.1f}%')
print(f'    Max similarity   : {np.max(all_similarities):.1f}%')
print()
print('Next step: Run 08_RAG_LLM_Generation.ipynb with rag_prompts_semantic.json')
print('='*60)

SEMANTIC RAG PIPELINE COMPLETE
  ChromaDB location      : dataset/chroma_db/
  Documents indexed      : 1,261
  Patients with prompts  : 4,508 (100% coverage)
  Old coverage (ID match): 1,192 (26%)
  Output file            : rag_prompts_semantic.json

  Retrieval quality (similarity scores):
    Mean similarity  : 62.1%
    Median similarity: 62.1%
    Min similarity   : 52.2%
    Max similarity   : 67.0%

Next step: Run 08_RAG_LLM_Generation.ipynb with rag_prompts_semantic.json
